# Q4 Appendix: Cross-Network Transfer — All Metrics (tidy format)

This appendix reports the article-ready Q4 cross-network transfer tables in
**tidy/long format**: every table carries a single set of four metric columns —
**Accuracy, Precision, Recall, Macro-F1** (`f1_score`) — as raw **means over
runs**, and the previously-widened condition (transfer direction / network role)
becomes **one label column** on the left.

**Invariants (identical to `Q4/Q4_analysis.ipynb`):**
- Drop split `Random with same distribution`.
- Cross-network only (`train_set != test_set`).
- Exclude the SetA$\leftrightarrow$SetC pair (same physical network, different capture period).
- All 11 models; direction = `train_set` $\to$ `test_set`; average over `run_no`.

**Tidy rule:** each table pivots only `metric` into the four columns (Accuracy,
Precision, Recall, Macro-F1). All headline derived quantities are dropped:
asymmetry, "Better source", Source$-$Target, ranks, mean$|$asym$|$, max$|$asym$|$,
Best/Worst source, and all statistical machinery (Wilcoxon $p$, Cohen's $d$).
Macro-F1 values are unchanged from the previous wide tables.


## Setup and data loading

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from itertools import combinations
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = Path('../data/wandb_export_final_hyperparameters.csv')
TAB_DIR   = Path('tables'); TAB_DIR.mkdir(exist_ok=True)

# ── Q4 invariants ─────────────────────────────────────────────────────────────
NETWORKS   = ['SetA', 'SetB', 'SetC', 'SetD']
EXCLUDED_PAIRS     = {('SetA', 'SetC'), ('SetC', 'SetA')}   # directed
EXCLUDED_UNORDERED = {frozenset(('SetA', 'SetC'))}          # unordered
PAIRS      = [p for p in combinations(NETWORKS, 2)
              if frozenset(p) not in EXCLUDED_UNORDERED]    # 5 unordered pairs
ALL_MODELS = ['gru', 'knn', 'lightgbm', 'logistic-regression', 'lstm',
              'mlp', 'nn', 'random-forest', 'rnn', 'svm', 'xgboost']
TASKS   = ['Binary', 'Multiclass']

# The four metrics carried through, with display labels. Macro-F1 == f1_score.
METRICS      = ['accuracy', 'precision', 'recall', 'f1_score']
METRIC_LABEL = {'accuracy': 'Accuracy', 'precision': 'Precision',
                'recall': 'Recall', 'f1_score': 'Macro-F1'}
PRIMARY      = 'f1_score'   # headline derived quantity uses Macro-F1

PAPER_MODEL_NAMES = {
    'gru': 'GRU', 'lstm': 'LSTM', 'rnn': 'RNN', 'mlp': 'MLP', 'nn': 'NN',
    'knn': 'kNN', 'lightgbm': 'LightGBM', 'logistic-regression': 'LR',
    'random-forest': 'RandomForest', 'svm': 'SVM', 'xgboost': 'XGBoost',
}
CAPTION_NOTE = (r'Values are means over runs of Accuracy/Precision/Recall/Macro-F1. '
                r'The SetA$\leftrightarrow$SetC pair is excluded '
                r'(same network, different capture period).')


def drop_excluded_pairs(df, train_col='train_set', test_col='test_set'):
    keep = ~df.apply(lambda r: (r[train_col], r[test_col]) in EXCLUDED_PAIRS, axis=1)
    return df[keep].copy()


print('Setup ready. Pairs analysed:', [f'{a}-{b}' for a, b in PAIRS])


Setup ready. Pairs analysed: ['SetA-SetB', 'SetA-SetD', 'SetB-SetC', 'SetB-SetD', 'SetC-SetD']


The next cell loads the results CSV, melts **all four metrics** into a long format (carrying the `metric` column), keeps cross-network rows only, drops the excluded pair, then averages over runs and pivots so each configuration has one column per metric.

In [2]:
# ── Load raw data ─────────────────────────────────────────────────────────────
raw = pd.read_csv(DATA_PATH, index_col=0)
raw = raw[raw['split'] != "Random with same distribution"]

metric_cols = [c for c in raw.columns if c.count('/') == 2]
id_vars = ['model', 'task', 'split', 'enable_sequences', 'run_no']

long = raw[id_vars + metric_cols].melt(
    id_vars=id_vars, value_vars=metric_cols,
    var_name='metric_key', value_name='value'
)
long[['train_set', 'test_set', 'metric']] = long['metric_key'].str.split('/', expand=True)
long = long.drop(columns='metric_key')

# Cross-network rows only — ALL four metrics carried through (no PRIMARY filter).
cross = long[
    (long['train_set'] != long['test_set']) &
    (long['metric'].isin(METRICS))
].copy()
cross['direction'] = cross['train_set'] + u'→' + cross['test_set']
cross = drop_excluded_pairs(cross)   # exclude SetA<->SetC

print(f'Cross-network rows (all metrics): {len(cross):,}')
print(f'Directed pairs: {cross["direction"].nunique()}')
print(f'Metrics carried: {sorted(cross["metric"].unique())}')

# ── Average across runs, one column per metric ────────────────────────────────
group_keys = ['model', 'task', 'split', 'enable_sequences',
              'train_set', 'test_set', 'direction']
avg_long = (cross.groupby(group_keys + ['metric'], as_index=False)['value']
                 .mean())
avg = (avg_long.pivot(index=group_keys, columns='metric', values='value')
               .reset_index())
avg.columns.name = None
print('Averaged cross-network configs:', len(avg))
print(avg[group_keys + METRICS].head(4).to_string(index=False))


Cross-network rows (all metrics): 24,080
Directed pairs: 10
Metrics carried: ['accuracy', 'f1_score', 'precision', 'recall']
Averaged cross-network configs: 680
model   task        split  enable_sequences train_set test_set direction  accuracy  precision   recall  f1_score
  gru Binary Random split              True      SetA     SetB SetA→SetB  0.935300   0.528152 0.519929  0.522501
  gru Binary Random split              True      SetA     SetD SetA→SetD  0.921661   0.805699 0.687819  0.726382
  gru Binary Random split              True      SetB     SetA SetB→SetA  0.645426   0.455472 0.375357  0.403561
  gru Binary Random split              True      SetB     SetC SetB→SetC  0.990054   0.964180 0.869610  0.908393


## Table q4_appendix_pair_asymmetry

Per-directed-transfer means. Each unordered pair contributes **two rows** (both
directions); the `Direction` label holds the directed transfer string (e.g.
`SetA→SetB`). Columns: `[Pair, Direction, Accuracy, Precision, Recall, Macro-F1]`.
Pairs are kept in fixed `PAIRS` order. The asymmetry and "Better source" derived
columns are dropped; the two `Direction` rows still recover them (fwd Macro-F1
$-$ rev Macro-F1). *(Source: `table1_asymmetry_per_pair`.)*


In [3]:
# ── Table 1: per-directed-transfer means per pair (tidy) ──────────────────────
pair_means = (avg.groupby(['train_set', 'test_set'], as_index=False)[METRICS].mean())

records = []
for net1, net2 in PAIRS:
    for src, tgt in ((net1, net2), (net2, net1)):
        row = pair_means[(pair_means['train_set'] == src) &
                         (pair_means['test_set'] == tgt)]
        if len(row) == 0:
            continue
        rec = {'Pair': f'{net1} ↔ {net2}', 'Direction': f'{src}→{tgt}'}
        for m in METRICS:
            rec[METRIC_LABEL[m]] = round(float(row[m].values[0]), 4)
        records.append(rec)

t1 = pd.DataFrame(records, columns=['Pair', 'Direction'] +
                  [METRIC_LABEL[m] for m in METRICS])

print('q4_appendix_pair_asymmetry')
print(t1.to_string(index=False))
t1.to_csv(TAB_DIR / 'q4_appendix_pair_asymmetry.csv', index=False)

latex1 = t1.to_latex(index=False, float_format='%.4f', escape=False,
    caption=(r'Per-directed-transfer cross-network results per network pair, all '
             r'metrics. Each unordered pair contributes both directed transfers '
             r'(Direction), reported as means over models, tasks, splits, input '
             r'representations, and runs. ' + CAPTION_NOTE),
    label='tab:q4_appendix_pair_asymmetry')
(TAB_DIR / 'q4_appendix_pair_asymmetry.tex').write_text(latex1)
print('Saved q4_appendix_pair_asymmetry.{csv,tex}')
t1


q4_appendix_pair_asymmetry
       Pair Direction  Accuracy  Precision  Recall  Macro-F1
SetA ↔ SetB SetA→SetB    0.8485     0.5761  0.4781    0.4825
SetA ↔ SetB SetB→SetA    0.7471     0.4802  0.4679    0.4384
SetA ↔ SetD SetA→SetD    0.8569     0.7739  0.6387    0.6464
SetA ↔ SetD SetD→SetA    0.5792     0.4558  0.5531    0.4030
SetB ↔ SetC SetB→SetC    0.8212     0.7235  0.6393    0.6194
SetB ↔ SetC SetC→SetB    0.8490     0.7680  0.7353    0.7017
SetB ↔ SetD SetB→SetD    0.7909     0.7418  0.6057    0.5747
SetB ↔ SetD SetD→SetB    0.6310     0.4750  0.5619    0.4138
SetC ↔ SetD SetC→SetD    0.8207     0.7625  0.6296    0.6150
SetC ↔ SetD SetD→SetC    0.6286     0.4857  0.5574    0.4284
Saved q4_appendix_pair_asymmetry.{csv,tex}


,Pair,Direction,Accuracy,Precision,Recall,Macro-F1
0,SetA ↔ SetB,SetA→SetB,0.8485,0.5761,0.4781,0.4825
1,SetA ↔ SetB,SetB→SetA,0.7471,0.4802,0.4679,0.4384
2,SetA ↔ SetD,SetA→SetD,0.8569,0.7739,0.6387,0.6464
3,SetA ↔ SetD,SetD→SetA,0.5792,0.4558,0.5531,0.4030
4,SetB ↔ SetC,SetB→SetC,0.8212,0.7235,0.6393,0.6194
5,SetB ↔ SetC,SetC→SetB,0.8490,0.7680,0.7353,0.7017
6,SetB ↔ SetD,SetB→SetD,0.7909,0.7418,0.6057,0.5747
7,SetB ↔ SetD,SetD→SetB,0.6310,0.4750,0.5619,0.4138
8,SetC ↔ SetD,SetC→SetD,0.8207,0.7625,0.6296,0.6150
9,SetC ↔ SetD,SetD→SetC,0.6286,0.4857,0.5574,0.4284


## Table q4_appendix_source_target

Per network, one row per **role**: the `Role` label is either `Source` (network
used as training source) or `Target` (network used as evaluation target).
Columns: `[Network, Role, Accuracy, Precision, Recall, Macro-F1]`. Networks are
ordered by their **Source Macro-F1 descending**; within each network the roles
follow the fixed order `Source`, `Target`. Ranks and the Source$-$Target
difference are dropped. *(Source: `table2_source_target_ranking`.)*


In [4]:
# ── Table 2: source / target network role (tidy) ──────────────────────────────
src_means = (avg.groupby('train_set', as_index=False)[METRICS].mean()
               .rename(columns={'train_set': 'Network'}))
tgt_means = (avg.groupby('test_set', as_index=False)[METRICS].mean()
               .rename(columns={'test_set': 'Network'}))

# Network order: by Source Macro-F1 descending.
net_order = (src_means.set_index('Network')[PRIMARY]
             .sort_values(ascending=False).index.tolist())

records = []
for net in net_order:
    for role, means in (('Source', src_means), ('Target', tgt_means)):
        row = means[means['Network'] == net]
        rec = {'Network': net, 'Role': role}
        for m in METRICS:
            rec[METRIC_LABEL[m]] = round(float(row[m].values[0]), 4)
        records.append(rec)

t2 = pd.DataFrame(records, columns=['Network', 'Role'] +
                  [METRIC_LABEL[m] for m in METRICS])

print('q4_appendix_source_target')
print(t2.to_string(index=False))
t2.to_csv(TAB_DIR / 'q4_appendix_source_target.csv', index=False)

latex2 = t2.to_latex(index=False, float_format='%.4f', escape=False,
    caption=(r'Per-network cross-network results by role as training source and '
             r'evaluation target, all metrics (averaged over all models, tasks, '
             r'splits, input representations, and runs). Networks ordered by '
             r'Source Macro-F1 descending. ' + CAPTION_NOTE),
    label='tab:q4_appendix_source_target')
(TAB_DIR / 'q4_appendix_source_target.tex').write_text(latex2)
print('Saved q4_appendix_source_target.{csv,tex}')
t2


q4_appendix_source_target
Network   Role  Accuracy  Precision  Recall  Macro-F1
   SetC Source    0.8349     0.7653  0.6825    0.6584
   SetC Target    0.7249     0.6046  0.5983    0.5239
   SetA Source    0.8527     0.6750  0.5584    0.5644
   SetA Target    0.6632     0.4680  0.5105    0.4207
   SetB Source    0.7864     0.6485  0.5710    0.5441
   SetB Target    0.7762     0.6064  0.5918    0.5327
   SetD Source    0.6129     0.4722  0.5575    0.4150
   SetD Target    0.8228     0.7594  0.6247    0.6120
Saved q4_appendix_source_target.{csv,tex}


,Network,Role,Accuracy,Precision,Recall,Macro-F1
0,SetC,Source,0.8349,0.7653,0.6825,0.6584
1,SetC,Target,0.7249,0.6046,0.5983,0.5239
2,SetA,Source,0.8527,0.6750,0.5584,0.5644
3,SetA,Target,0.6632,0.4680,0.5105,0.4207
4,SetB,Source,0.7864,0.6485,0.5710,0.5441
5,SetB,Target,0.7762,0.6064,0.5918,0.5327
6,SetD,Source,0.6129,0.4722,0.5575,0.4150
7,SetD,Target,0.8228,0.7594,0.6247,0.6120


## Table q4_appendix_per_model

Per model: mean **cross-network** metrics. Columns: `[model, Accuracy, Precision,
Recall, Macro-F1]`, sorted by **Macro-F1 descending**. This table already had a
single metric set (Mean Cross-net); the Best/Worst source and mean$|$asym$|$ /
max$|$asym$|$ derived columns are dropped. *(Source: `table3_per_model_asymmetry`.)*


In [5]:
# ── Table 3: per-model mean cross-network metrics (tidy) ──────────────────────
model_records = []
for model in ALL_MODELS:
    sub = avg[avg['model'] == model]
    rec = {'model': PAPER_MODEL_NAMES.get(model, model)}
    for m in METRICS:
        rec[METRIC_LABEL[m]] = round(sub[m].mean(), 4)
    model_records.append(rec)

t3 = (pd.DataFrame(model_records, columns=['model'] +
                   [METRIC_LABEL[m] for m in METRICS])
        .sort_values(METRIC_LABEL[PRIMARY], ascending=False)
        .reset_index(drop=True))

print('q4_appendix_per_model')
print(t3.to_string(index=False))
t3.to_csv(TAB_DIR / 'q4_appendix_per_model.csv', index=False)

latex3 = t3.to_latex(index=False, float_format='%.4f', escape=False,
    caption=(r'Per-model mean cross-network results, all metrics. Values are '
             r'means over all directed cross-network transfers, tasks, splits, '
             r'input representations, and runs; sorted by Macro-F1 descending. '
             + CAPTION_NOTE),
    label='tab:q4_appendix_per_model')
(TAB_DIR / 'q4_appendix_per_model.tex').write_text(latex3)
print('Saved q4_appendix_per_model.{csv,tex}')
t3


q4_appendix_per_model
       model  Accuracy  Precision  Recall  Macro-F1
         GRU    0.8586     0.6789  0.6429    0.5979
RandomForest    0.7872     0.6664  0.6364    0.5940
    LightGBM    0.7950     0.6650  0.6295    0.5939
     XGBoost    0.7929     0.6671  0.6235    0.5897
         RNN    0.8508     0.6735  0.6370    0.5853
         MLP    0.8375     0.6842  0.5876    0.5592
          LR    0.7973     0.6393  0.5710    0.5582
        LSTM    0.7574     0.5951  0.5863    0.4992
         kNN    0.6539     0.5406  0.5367    0.4577
          NN    0.6917     0.5686  0.5362    0.4337
         SVM    0.6129     0.5276  0.4950    0.3936
Saved q4_appendix_per_model.{csv,tex}


,model,Accuracy,Precision,Recall,Macro-F1
0,GRU,0.8586,0.6789,0.6429,0.5979
1,RandomForest,0.7872,0.6664,0.6364,0.5940
2,LightGBM,0.7950,0.6650,0.6295,0.5939
3,XGBoost,0.7929,0.6671,0.6235,0.5897
4,RNN,0.8508,0.6735,0.6370,0.5853
5,MLP,0.8375,0.6842,0.5876,0.5592
6,LR,0.7973,0.6393,0.5710,0.5582
7,LSTM,0.7574,0.5951,0.5863,0.4992
8,kNN,0.6539,0.5406,0.5367,0.4577
9,NN,0.6917,0.5686,0.5362,0.4337


## Table q4_appendix_pair_task_split

Per-(pair, task, split) directional means in tidy form. Each (pair, task, split)
group expands into **two rows** keyed by the `Direction` label (the directed
transfer string, e.g. `SetA→SetD`). Columns: `[pair, task, split, Direction,
Accuracy, Precision, Recall, Macro-F1]`. Directions are matched per (task) on
identical configs (model, split, `enable_sequences`) — the same paired-directions
logic as the source — then plain directional means per metric are reported. The
Asymmetry column is dropped; the source (pair, task, split) row order is
preserved. *(Source: `q4_tab_pair_asymmetry` / `table4_statistical_tests` granularity.)*


In [6]:
# ── Table 4: per-(pair, task, split) directional means (tidy) ─────────────────
# Match forward/reverse directions on identical configurations (model,
# split, enable_sequences); report plain directional means per metric, with
# the directed transfer as the `Direction` label (two rows per group).
cfg_keys = ['model', 'split', 'enable_sequences']

def paired_directions(sub, net1, net2, keys):
    fwd = sub[(sub['train_set'] == net1) & (sub['test_set'] == net2)]
    rev = sub[(sub['train_set'] == net2) & (sub['test_set'] == net1)]
    fwd = fwd[keys + METRICS].rename(columns={m: f'{m}_fwd' for m in METRICS})
    rev = rev[keys + METRICS].rename(columns={m: f'{m}_rev' for m in METRICS})
    return fwd.merge(rev, on=keys, how='inner')

rows = []
for net1, net2 in PAIRS:
    for task in TASKS:
        sub = avg[avg['task'] == task]
        m = paired_directions(sub, net1, net2, cfg_keys)
        if len(m) < 2:
            continue
        for split, g in m.groupby('split'):
            base = {'pair': f'{net1}↔{net2}', 'task': task, 'split': split}
            # abs Macro-F1 asymmetry retained only to reproduce source ordering.
            abs_asym = abs(g[f'{PRIMARY}_fwd'].mean() - g[f'{PRIMARY}_rev'].mean())
            for direction, suffix, src, tgt in (
                    (f'{net1}→{net2}', 'fwd', net1, net2),
                    (f'{net2}→{net1}', 'rev', net2, net1)):
                rec = dict(base)
                rec['Direction'] = direction
                for me in METRICS:
                    rec[METRIC_LABEL[me]] = round(g[f'{me}_{suffix}'].mean(), 4)
                rec['_abs'] = abs_asym
                rows.append(rec)

pts = pd.DataFrame(rows)
# Preserve source ordering: pairs by mean |MacroF1 asym| desc, then task, split.
pair_order = (pts.groupby('pair')['_abs'].mean()
              .sort_values(ascending=False).index.tolist())
pts['pair'] = pd.Categorical(pts['pair'], categories=pair_order, ordered=True)
pts = (pts.sort_values(['pair', 'task', 'split'])
          .drop(columns='_abs').reset_index(drop=True))
pts = pts[['pair', 'task', 'split', 'Direction'] +
          [METRIC_LABEL[m] for m in METRICS]]

print('q4_appendix_pair_task_split')
print(pts.to_string(index=False))
pts.to_csv(TAB_DIR / 'q4_appendix_pair_task_split.csv', index=False)

latex4 = pts.to_latex(index=False, float_format='%.4f', escape=False,
    caption=(r'Per-(pair, task, split) directional means of cross-network '
             r'transfer, all metrics. For each unordered pair and task, forward '
             r'and reverse directions are matched on identical configurations '
             r'(model, split, input representation); reported values are plain '
             r'directional means per metric, one row per directed transfer '
             r'(Direction). ' + CAPTION_NOTE),
    label='tab:q4_appendix_pair_task_split')
(TAB_DIR / 'q4_appendix_pair_task_split.tex').write_text(latex4)
print('Saved q4_appendix_pair_task_split.{csv,tex}')
pts


q4_appendix_pair_task_split
     pair       task        split Direction  Accuracy  Precision  Recall  Macro-F1
SetA↔SetD     Binary Random split SetA→SetD    0.8707     0.8231  0.7093    0.7280
SetA↔SetD     Binary Random split SetD→SetA    0.5375     0.5160  0.5056    0.4296
SetA↔SetD     Binary   Time split SetA→SetD    0.8593     0.8625  0.6964    0.6879
SetA↔SetD     Binary   Time split SetD→SetA    0.6448     0.5493  0.6600    0.4853
SetA↔SetD Multiclass Random split SetA→SetD    0.8382     0.7301  0.5641    0.5903
SetA↔SetD Multiclass Random split SetD→SetA    0.4888     0.3656  0.4609    0.3162
SetA↔SetD Multiclass   Time split SetA→SetD    0.8595     0.6800  0.5850    0.5793
SetA↔SetD Multiclass   Time split SetD→SetA    0.6456     0.3922  0.5861    0.3808
SetC↔SetD     Binary Random split SetC→SetD    0.8104     0.8024  0.6970    0.6811
SetC↔SetD     Binary Random split SetD→SetC    0.6721     0.5635  0.6936    0.5152
SetC↔SetD     Binary   Time split SetC→SetD    0.8695     0

,pair,task,split,Direction,Accuracy,Precision,Recall,Macro-F1
0,SetA↔SetD,Binary,Random split,SetA→SetD,0.8707,0.8231,0.7093,0.7280
1,SetA↔SetD,Binary,Random split,SetD→SetA,0.5375,0.5160,0.5056,0.4296
2,SetA↔SetD,Binary,Time split,SetA→SetD,0.8593,0.8625,0.6964,0.6879
3,SetA↔SetD,Binary,Time split,SetD→SetA,0.6448,0.5493,0.6600,0.4853
4,SetA↔SetD,Multiclass,Random split,SetA→SetD,0.8382,0.7301,0.5641,0.5903
5,SetA↔SetD,Multiclass,Random split,SetD→SetA,0.4888,0.3656,0.4609,0.3162
6,SetA↔SetD,Multiclass,Time split,SetA→SetD,0.8595,0.6800,0.5850,0.5793
7,SetA↔SetD,Multiclass,Time split,SetD→SetA,0.6456,0.3922,0.5861,0.3808
8,SetC↔SetD,Binary,Random split,SetC→SetD,0.8104,0.8024,0.6970,0.6811
9,SetC↔SetD,Binary,Random split,SetD→SetC,0.6721,0.5635,0.6936,0.5152
